### 5. Feature Engineering
**Mục tiêu cốt lõi:**
- Tạo ra các biến mới để mô hình bắt được các quy luật mua sắm phức tạp
- Loại bỏ các cột không mang thông tin để giảm nhiễu
- Giảm chiều dữ liệu và chuẩn hóa thang đo giúp thuật toán hội tụ nhanh hơn
- Giúp mô hình hoạt động tốt trên tập Test và dữ liệu thực tế

Các bước thực hiện:
- Data split: Chia tập train/test theo tỷ lệ sample 80/20
- Feature Extraction:
    - Date features: Tạo biến Is_Promotion_Campaign (1 nếu Tháng 1 hoặc Tháng 2, ngược lại 0)
- Feature Transformation:
    - Log-Transform: Biến mục tiêu price (Doanh thu) bị lệch phải nghiêm trọng cần biến đối logarit để kéo phân phối về hình chuông, giúp thỏa mãn giả định của Linear Regression
    - Rời rạc hóa (Binning): Chuyển đổi cột age (tuổi từ 18-90) thành các nhóm (Bins) như: Gen Z (<25), Millennials (25-40), Gen X (41-60), Boomers (>60). Việc này giúp mô hình dễ dàng học được sự khác biệt về chi tiêu giữa các thế hệ thay vì coi tuổi tác là một đường thẳng tuyến tính
- Feature Selection:
    - Loại bỏ invoice_no, customer_id, invoice_date (đã trích xuất), và 3 biến phân loại nhiễu (shopping_mall, gender, payment_method)
- Encoding Categorical Variables:
    - Gom category và age_group vào bộ One-Hot Encoding (OHE) để bẻ thành ma trận nhị phân
- Scaling (Chuẩn hóa thang đo): Áp dụng Standardization (Z-score) cho các biến số liên tục để kéo Mean = 0 và Std = 1. Việc này giúp thuật toán Gradient Descent trong Ridge Regression không bị chệch hướng bởi các biến có giá trị quá lớn

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('./data/preprocessed-data/preprocessed_customer_shopping_data.csv', parse_dates=['invoice_date'])

In [3]:
# Tách Feature (X) và Target (y)
X = data.drop(columns=['price'])
y = data['price']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")
print("\nCác cột features còn lại trong X_train:")
print(X_train.columns.tolist())

Kích thước X_train: (79565, 9)
Kích thước X_test: (19892, 9)

Các cột features còn lại trong X_train:
['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity', 'payment_method', 'invoice_date', 'shopping_mall']


In [5]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 79565 entries, 59044 to 15795
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   invoice_no      79565 non-null  object        
 1   customer_id     79565 non-null  object        
 2   gender          79565 non-null  object        
 3   age             79565 non-null  int64         
 4   category        79565 non-null  object        
 5   quantity        79565 non-null  int64         
 6   payment_method  79565 non-null  object        
 7   invoice_date    79565 non-null  datetime64[ns]
 8   shopping_mall   79565 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(6)
memory usage: 6.1+ MB


In [6]:
def extract_features(X_data):
    """Trích xuất đặc trưng thời gian"""
    X = X_data.copy()
    X['Is_Promotion_Campaign'] = X['invoice_date'].dt.month.isin([1, 2]).astype(int)
    return X

In [7]:
def bin_age(X_data):
    """Rời rạc hóa (Binning) nhóm tuổi"""
    X = X_data.copy()
    bins = [0, 25, 40, 60, 100]
    labels = ['Gen Z', 'Millennials', 'Gen X', 'Boomers']
    X['Age_Group'] = pd.cut(X['age'], bins=bins, labels=labels)
    # Xóa cột age gốc để tránh đa cộng tuyến
    return X.drop(columns=['age'])

In [8]:
X_train = extract_features(X_train)
X_test = extract_features(X_test)

X_train = bin_age(X_train)
X_test = bin_age(X_test)

In [9]:
# Log-Transform cho biến mục tiêu y (Khắc phục phân phối lệch phải)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [11]:
cols_to_drop = ['invoice_no', 'customer_id', 'invoice_date', 'shopping_mall', 'gender', 'payment_method']
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

In [22]:
X_train.head()

,category,quantity,Is_Promotion_Campaign,Age_Group
59044,Toys,1,0,Gen Z
69682,Souvenir,1,0,Boomers
79039,Shoes,1,1,Boomers
87384,Shoes,5,0,Millennials
808,Cosmetics,2,1,Millennials


In [ ]:
numeric_features = ['quantity']

# Các biến chữ -> Cần One-Hot Encoding
categorical_features = ['category','Age_Group']

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features) # sparse_output=False để trả về ma trận đặc
    ],
    remainder='passthrough'
)

In [15]:
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

In [17]:
print(f"   - Tập X_train_final: {X_train_final.shape}")
print(f"   - Tập X_test_final: {X_test_final.shape}")

   - Tập X_train_final: (79565, 14)
   - Tập X_test_final: (19892, 14)


In [18]:
os.makedirs('./data/ready_for_train', exist_ok=True)

joblib.dump(X_train_final, './data/ready_for_train/X_train_final.pkl')
joblib.dump(X_test_final, './data/ready_for_train/X_test_final.pkl')

joblib.dump(y_train_log, './data/ready_for_train/y_train_log.pkl')
joblib.dump(y_test_log, './data/ready_for_train/y_test_log.pkl')

# Dành cho Model Deployment
joblib.dump(preprocessor, './data/ready_for_train/preprocessor.pkl')

['./data/ready_for_train/preprocessor.pkl']